# Imports

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager 
from tqdm import tqdm
from bs4 import BeautifulSoup
import time 
import os
from pathlib import Path 

In [2]:
bible_languages_url = {
    "english": "https://www.bible.com/bible/111/GEN.1.NIV",              #1
    "cebuano": "https://www.bible.com/bible/2187/GEN.1.ABCEB",       #14
    "waray": "https://www.bible.com/bible/2198/GEN.1.MBBSAM",           #15
}

bible_books = ["GEN", "EXO", "LEV", "NUM", "DEU", 
               "JOS", "JDG", "RUT", "1SA", "2SA",
                "1KI", "2KI", "1CH", "2CH", "EZR",
                "NEH", "EST", "JOB", "PSA", "PRO",
                "ECC", "SNG", "ISA", "JER", "LAM",
                "EZK", "DAN", "HOS", "JOL", "AMO",
                "OBA", "JON", "MIC", "NAM", "HAB",
                "ZEP", "HAG", "ZEC", "MAL",
               "MAT", "MRK", "LUK", "JHN", "ACT", 
               "ROM", "1C0", "2CO", "GAL", "EPH", 
               "PHP", "COL", "1TH", "2TH", "1TI", 
               "2TI", "TIT", "PHM", "HEB", "JAS", 
               "1PE", "2PE", "1JN", "2JN", "3JN", 
               "JUD", "REV"
               ]

In [3]:
print("Setting up WebDriver...")
options = Options()
options.add_argument("--headless")  # run chrome without opening a visual window
options.add_argument("--log-level=3")  # suppress unnecessary logs
options.add_experimental_option('excludeSwitches', ['enable-logging'])

# use WebDriver Manager to handle driver installation/updates automatically
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
print("WebDriver ready.")

# data structures for storing scraped data and statistics
bible_data_new = {}  # {lang: {book: {chapter: {verse_num: text}}}}
word_counts_new = {}  # {lang: count}
total_words_new = 0
total_verses_new = 0
total_chapters_new = 0

output_dir = Path("../data/raw")
output_dir.mkdir(parents=True, exist_ok=True)

# Determine which books to scrape for each language
books_to_scrape = {}
for lang in bible_languages_url.keys():
    output_file = output_dir / f"{lang}_raw.txt"
    existing_books = set()
    if output_file.exists():
        with open(output_file, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    book_code = line.split()[0]
                    existing_books.add(book_code)
    # Only keep books not already present
    books_to_scrape[lang] = [book for book in bible_books if book not in existing_books]

# progress bar setup
total_iterations = sum(len(books) for books in books_to_scrape.values())
pbar = tqdm(total=total_iterations, desc="Overall Progress", unit="book")


try:
    for lang, root_url in bible_languages_url.items():
        bible_data_new[lang] = {}
        word_counts_new[lang] = 0

        # extract the parts of the URL
        try:
            parts = root_url.split("/")
            base_bible_url = f"https://www.bible.com/bible/{parts[4]}"
            version = root_url.split(".")[-1]
        except IndexError:
            print(f"Skipping invalid URL format for {lang}: {root_url}")
            pbar.update(len(bible_books))  
            continue  # skip to next language if error occurs

        # Only scrape books not already present
        for book in books_to_scrape[lang]:
            pbar.set_description(f"Scraping {lang} - {book}")
            bible_data_new[lang][book] = {}

            ch = 1
            while True:
                url = f"{base_bible_url}/{book}.{ch}.{version}"
                driver.get(url)
                time.sleep(1)  # wait for page to load

                soup = BeautifulSoup(driver.page_source, "html.parser")
                
                # checks if end of chapters reached using "not available" marker
                not_available = soup.find("span", class_="ChapterContent_not-avaliable-span__WrOM_")
                if not_available:
                    break

                # if not yet end of chapters, extract
                chapter_content = soup.find_all("span", {"data-usfm": True})

                # mark as empty if no content found
                if not chapter_content:
                    bible_data_new[lang][book][ch] = {}
                else:
                    chapter_word_count_new = 0

                    # Group elements by data-usfm to handle duplicate verse codes
                    verse_groups = {}
                    for verse in chapter_content:
                        usfm = verse.get("data-usfm")
                        if usfm not in verse_groups:
                            verse_groups[usfm] = []
                        verse_groups[usfm].append(verse)

                    # Process each verse group
                    verse_dict = {}  # {verse_num: text}
                    
                    for usfm, verse_elements in verse_groups.items():
                        # Skip grouped verses (e.g., "GEN.1.1-3" or "GEN.1.1,3")
                        if '-' in usfm or ',' in usfm:
                            continue
                        
                        # Extract verse number from usfm (e.g., "GEN.1.1" -> 1)
                        try:
                            verse_num = int(usfm.split('.')[-1])
                        except (ValueError, IndexError):
                            continue
                        
                        combined_text_parts = []
                        
                        for verse in verse_elements:
                            # remove footnotes within the verse
                            for note in verse.find_all("span", class_=lambda x: x and x.startswith("ChapterContent_note")):
                                note.decompose()

                            # extract clean verse text
                            verse_text = verse.get_text(" ", strip=True)
                            if verse_text:
                                combined_text_parts.append(verse_text)
                        
                        # Combine all parts of the same verse
                        if combined_text_parts:
                            combined_verse_text = " ".join(combined_text_parts)
                            verse_dict[verse_num] = combined_verse_text
                            verse_words_new = len(combined_verse_text.split())
                            chapter_word_count_new += verse_words_new
                            total_verses_new += 1

                    # Store verses with actual verse numbers as keys
                    bible_data_new[lang][book][ch] = verse_dict

                    # update word counts
                    word_counts_new[lang] += chapter_word_count_new
                    total_words_new += chapter_word_count_new
                    total_chapters_new += 1

                    pbar.set_postfix({
                        'Words': f"{total_words_new:,}",
                        'Verses': f"{total_verses_new:,}",
                        'Chapters': total_chapters_new
                    })

                ch += 1       # next chapter
                if ch > 100:  # safety cap
                    break

            pbar.update(1)

finally:
    driver.quit()
    pbar.close()
    print("Scraping complete.")

# summary statistics
print("\n" + "="*60)
if total_verses_new > 0 and total_chapters_new > 0:
    print(f"Total Words: {total_words_new:,}")
    print(f"Total Verses: {total_verses_new:,}")
    print(f"Total Chapters: {total_chapters_new}")
    print(f"Average Words per Verse: {total_words_new/total_verses_new:.1f}")
    print(f"Average Words per Chapter: {total_words_new/total_chapters_new:.1f}")
else:
    print("No data scraped or processed.")

print("\nWord Count by Language:")
print("-" * 30)
if total_words_new > 0:
    for lang_key, count in word_counts_new.items():
        if count > 0:
            percentage = (count / total_words_new) * 100
            print(f"{lang_key:12}: {count:8,} words ({percentage:.1f}%)")
        else:
            print(f"{lang_key:12}: {count:8,} words (0.0%) - Check availability/URL")
else:
    print("No words counted.")

Setting up WebDriver...
WebDriver ready.


Scraping waray - 1C0: 100%|██████████| 90/90 [46:39<00:00, 31.10s/book, Words=1,076,743, Verses=40,134, Chapters=1764] 

Scraping complete.

Total Words: 1,076,743
Total Verses: 40,134
Total Chapters: 1764
Average Words per Verse: 26.8
Average Words per Chapter: 610.4

Word Count by Language:
------------------------------
english     :  336,402 words (31.2%)
cebuano     :  377,267 words (35.0%)
waray       :  363,074 words (33.7%)


In [4]:
bible_data_new

{'english': {'1KI': {1: {1: '1 When King David was very old, he could not keep warm even when they put covers over him.',
    2: '2 So his attendants said to him, “Let us look for a young virgin to serve the king and take care of him. She can lie beside him so that our lord the king may keep warm.”',
    3: '3 Then they searched throughout Israel for a beautiful young woman and found Abishag, a Shunammite, and brought her to the king.',
    4: '4 The woman was very beautiful; she took care of the king and waited on him, but the king had no sexual relations with her.',
    5: '5 Now Adonijah, whose mother was Haggith, put himself forward and said, “I will be king.” So he got chariots and horses ready, with fifty men to run ahead of him.',
    6: '6 (His father had never rebuked him by asking, “Why do you behave as you do?” He was also very handsome and was born next after Absalom.)',
    7: '7 Adonijah conferred with Joab son of Zeruiah and with Abiathar the priest, and they gave him th

In [5]:
# Save bible data to text files
for lang, books in bible_data_new.items():
    output_file = output_dir / f"{lang}_raw.txt"
    with open(output_file, "a", encoding="utf-8") as f:
        for book, chapters in books.items():
            for chapter_num, verses in chapters.items():
                for verse_num in sorted(verses.keys()):
                    verse_text = verses[verse_num]
                    f.write(f"{book} {chapter_num}:{verse_num} {verse_text}\n")
        print(f"Appended new books for {lang} to {output_file}")

print("\nAll new books scraped and appended successfully!")

Appended new books for english to ..\data\raw\english_raw.txt
Appended new books for cebuano to ..\data\raw\cebuano_raw.txt
Appended new books for waray to ..\data\raw\waray_raw.txt

All new books scraped and appended successfully!
